In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/siim-isic-melanoma-classification/sample_submission.csv
/kaggle/input/siim-isic-melanoma-classification/train.csv
/kaggle/input/siim-isic-melanoma-classification/test.csv
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_2417927.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_0546632.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_2227618.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_7969975.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_3356592.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_8170270.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_2799021.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_4693305.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_3783148.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_9729657.jpg
/kaggle/input/siim-isic-melanoma-classification/jpeg/test/ISIC_17

In [14]:
import os
os.listdir('/kaggle/input')


['siim-isic-melanoma-classification']

In [15]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, models, transforms
from torchvision.models import MobileNet_V3_Large_Weights
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything()

print("✅ Imports & seeds ready")

✅ Imports & seeds ready


In [26]:
IMG_SIZE = 384
BATCH_SIZE = 32            # effective = 32 × 2 GPUs = 64
EPOCHS = 10
LR = 3e-4
PATIENCE = 5               # Early stopping patience

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()

print(f"🚀 Device: {DEVICE}")
print(f"🔥 GPUs available: {NUM_GPUS}")

🚀 Device: cuda
🔥 GPUs available: 1


In [27]:


TRAIN_CSV = "/kaggle/input/siim-isic-melanoma-classification/train.csv"
TRAIN_IMG_DIR = "/kaggle/input/siim-isic-melanoma-classification/jpeg/train"

df = pd.read_csv(TRAIN_CSV)
print(f"📊 Samples: {len(df)}")
print(df['target'].value_counts())

📊 Samples: 33126
target
0    32542
1      584
Name: count, dtype: int64


In [28]:
class MelanomaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row.image_name}.jpg")

        try:
            img = Image.open(img_path).convert("RGB")
        except:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128,128,128))

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(row.target, dtype=torch.long)
        return img, label

In [29]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.1,0.1,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [30]:
# ===============================
# TRAIN / VALIDATION SPLIT
# ===============================

from sklearn.model_selection import train_test_split

# Load CSV if not already loaded
if 'df' not in globals():
    df = pd.read_csv(TRAIN_CSV)

# Make sure target column exists
assert 'target' in df.columns, "❌ 'target' column not found in CSV"

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['target'],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("✅ train_df and val_df created")
print("Train size:", len(train_df))
print("Val size:", len(val_df))
print(train_df['target'].value_counts())

✅ train_df and val_df created
Train size: 26500
Val size: 6626
target
0    26033
1      467
Name: count, dtype: int64


In [31]:
class MelanomaNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.mobilenet_v3_large(
            weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2
        )
        in_features = self.model.classifier[0].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        return self.model(x)

model = MelanomaNet().to(DEVICE)

# 🔥 MULTI-GPU
if NUM_GPUS > 1:
    model = nn.DataParallel(model)

print("✅ Model ready (Multi-GPU enabled)")

✅ Model ready (Multi-GPU enabled)


In [32]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scaler = torch.cuda.amp.GradScaler()

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=5, T_mult=2
)

In [33]:
def train_epoch(loader):
    model.train()
    total, correct, loss_sum = 0, 0, 0

    for x,y in tqdm(loader, leave=False):
        x,y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item()*y.size(0)
        correct += (out.argmax(1)==y).sum().item()
        total += y.size(0)

    return loss_sum/total, 100*correct/total


def validate_epoch(loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0

    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out,y)

            loss_sum += loss.item()*y.size(0)
            correct += (out.argmax(1)==y).sum().item()
            total += y.size(0)

    return loss_sum/total, 100*correct/total

In [34]:
# ===============================
# FIX CELL — ENSURE DATALOADERS EXIST
# ===============================

assert 'train_df' in globals(), "❌ train_df missing (run split cell)"
assert 'val_df' in globals(), "❌ val_df missing (run split cell)"
assert 'train_tfms' in globals(), "❌ train_tfms missing"
assert 'val_tfms' in globals(), "❌ val_tfms missing"

train_ds = MelanomaDataset(train_df, TRAIN_IMG_DIR, train_tfms)
val_ds   = MelanomaDataset(val_df, TRAIN_IMG_DIR, val_tfms)

counts = train_df['target'].value_counts()
weights = train_df['target'].map({0:1.0, 1:counts[0]/counts[1]}).values

sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("✅ train_loader and val_loader are now defined")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")

✅ train_loader and val_loader are now defined
   Train batches: 829
   Val batches:   208


In [35]:
best_acc = 0.0
patience_counter = 0

train_history = []
val_history = []

print("\n🚀 Starting Training...\n")

for epoch in range(EPOCHS):
    print(f"🔁 Epoch {epoch+1}/{EPOCHS}")

    # --- TRAIN ---
    train_loss, train_acc = train_epoch(train_loader)

    # --- VALIDATE ---
    val_loss, val_acc = validate_epoch(val_loader)

    # Scheduler step AFTER validation
    scheduler.step()

    # Save history
    train_history.append((train_loss, train_acc))
    val_history.append((val_loss, val_acc))

    print(
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% || "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
    )

    # --- CHECKPOINT ---
    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0

        torch.save(
            model.module.state_dict() if isinstance(model, nn.DataParallel)
            else model.state_dict(),
            "best_model.pth"
        )

        print(f"💾 Best model saved (Val Acc = {best_acc:.2f}%)")

    else:
        patience_counter += 1
        print(f"⏳ EarlyStopping Counter: {patience_counter}/{PATIENCE}")

    # --- EARLY STOPPING ---
    if patience_counter >= PATIENCE:
        print("⛔ Early stopping triggered")
        break

print("\n🎯 Training Finished")
print(f"🔥 Best Validation Accuracy: {best_acc:.2f}%")


🚀 Starting Training...

🔁 Epoch 1/10


Train Loss: 0.2617 | Train Acc: 89.06% || Val Loss: 0.2777 | Val Acc: 90.30%
💾 Best model saved (Val Acc = 90.30%)
🔁 Epoch 2/10


Train Loss: 0.1119 | Train Acc: 95.94% || Val Loss: 0.1175 | Val Acc: 95.83%
💾 Best model saved (Val Acc = 95.83%)
🔁 Epoch 3/10


Train Loss: 0.0586 | Train Acc: 98.06% || Val Loss: 0.1021 | Val Acc: 97.31%
💾 Best model saved (Val Acc = 97.31%)
🔁 Epoch 4/10


Train Loss: 0.0271 | Train Acc: 99.17% || Val Loss: 0.0975 | Val Acc: 97.51%
💾 Best model saved (Val Acc = 97.51%)
🔁 Epoch 5/10


...

Train Loss: 0.0788 | Train Acc: 97.18% || Val Loss: 0.1258 | Val Acc: 96.85%
⏳ EarlyStopping Counter: 1/5
🔁 Epoch 7/10


Train Loss: 0.0482 | Train Acc: 98.44% || Val Loss: 0.1656 | Val Acc: 95.74%
⏳ EarlyStopping Counter: 2/5
🔁 Epoch 8/10


Train Loss: 0.0313 | Train Acc: 99.02% || Val Loss: 0.1089 | Val Acc: 97.34%
⏳ EarlyStopping Counter: 3/5
🔁 Epoch 9/10


Train Loss: 0.0277 | Train Acc: 99.08% || Val Loss: 0.1519 | Val Acc: 97.59%
⏳ EarlyStopping Counter: 4/5
🔁 Epoch 10/10


Train Loss: 0.0178 | Train Acc: 99.43% || Val Loss: 0.1245 | Val Acc: 97.90%
⏳ EarlyStopping Counter: 5/5
⛔ Early stopping triggered

🎯 Training Finished
🔥 Best Validation Accuracy: 97.95%


In [36]:
 from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, f1_score
)

# Load best model
if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(torch.load("best_model.pth"))
else:
    model.load_state_dict(torch.load("best_model.pth"))

model.eval()
print("✅ Best model loaded")

✅ Best model loaded


In [37]:
all_targets = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="🔍 Evaluating"):
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        outputs = model(images)
        probs = F.softmax(outputs, dim=1)[:, 1]  # malignant prob
        preds = torch.argmax(outputs, dim=1)

        all_targets.extend(targets.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

print("✅ Evaluation forward pass completed")

🔍 Evaluating: 100%|██████████| 208/208 [07:40<00:00,  2.21s/it]

✅ Evaluation forward pass completed


In [38]:
acc = accuracy_score(all_targets, all_preds)
precision = precision_score(all_targets, all_preds, zero_division=0)
recall = recall_score(all_targets, all_preds, zero_division=0)
f1 = f1_score(all_targets, all_preds, zero_division=0)
auc = roc_auc_score(all_targets, all_probs)
cm = confusion_matrix(all_targets, all_preds)

print("\n🎯 FINAL EVALUATION SUMMARY")
print("="*70)
print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"AUC-ROC   : {auc:.4f}")

print("\n📊 Confusion Matrix")
print(cm)

print("\n📋 Classification Report")
print(
    classification_report(
        all_targets,
        all_preds,
        target_names=["Benign", "Malignant"],
        digits=4
    )
)


🎯 FINAL EVALUATION SUMMARY
Accuracy  : 97.95%
Precision : 0.3582
Recall    : 0.2051
F1-score  : 0.2609
AUC-ROC   : 0.9024

📊 Confusion Matrix
[[6466   43]
 [  93   24]]

📋 Classification Report
              precision    recall  f1-score   support

      Benign     0.9858    0.9934    0.9896      6509
   Malignant     0.3582    0.2051    0.2609       117

    accuracy                         0.9795      6626
   macro avg     0.6720    0.5993    0.6252      6626
weighted avg     0.9747    0.9795    0.9767      6626

